In [1]:
import sys
from pathlib import Path

sys.path.insert(0, "../src")

import pandas as pd

from ufa import (
    prepare_all_games_training_data,
    train_etv_models,
    build_etv_model,
    save_model_bundle,
    build_game_throws,
)

In [2]:
training_throws = prepare_all_games_training_data("../data/raw/all_games_1024.csv")

print("Training games:", training_throws["gameID"].nunique())
print("Training throws:", len(training_throws))
print("Completion rate:", training_throws["completion"].mean().round(3))
print("Point outcome rate:", training_throws["point_outcome"].mean().round(3))

training_throws[[
    "gameID",
    "thrower",
    "receiver",
    "completion",
    "point_outcome",
    "throw_distance",
    "times",
]].head(20)

Training games: 574
Training throws: 290826
Completion rate: 0.932
Point outcome rate: 0.71


,gameID,thrower,receiver,completion,point_outcome,throw_distance,times
0,2023-06-24-DC-BOS,jnissen,jmalks,1,1,12.430326,2880.000000
1,2023-06-24-DC-BOS,jmalks,jnissen,1,1,10.948397,2876.578947
2,2023-06-24-DC-BOS,jnissen,jmalks,1,1,13.982221,2873.157895
3,2023-06-24-DC-BOS,jmalks,jnissen,1,1,9.227448,2869.736842
4,2023-06-24-DC-BOS,jnissen,boort,1,1,19.372426,2866.315789
5,2023-06-24-DC-BOS,boort,jmalks,1,1,11.366653,2862.894737
6,2023-06-24-DC-BOS,jmalks,jnissen,1,1,11.147242,2859.473684
7,2023-06-24-DC-BOS,jnissen,khealey,1,1,12.570143,2856.052632
8,2023-06-24-DC-BOS,khealey,jnissen,1,1,6.413774,2852.631579
9,2023-06-24-DC-BOS,jnissen,cboxley,1,1,11.741082,2849.210526


In [3]:
training_throws[[
    "thrower_x",
    "thrower_y",
    "receiver_x",
    "receiver_y",
    "throw_distance",
    "throw_angle",
    "y_diff",
    "x_diff",
    "times",
    "completion_target",
    "eventual_score_target",
]].describe()

,thrower_x,thrower_y,receiver_x,receiver_y,throw_distance,throw_angle,y_diff,x_diff,times,completion_target,eventual_score_target
count,290826.000000,290826.000000,290826.000000,290826.000000,290826.000000,290826.000000,290826.000000,290826.000000,290826.000000,290826.000000,290826.000000
mean,0.395374,51.250743,0.458543,59.614521,16.602010,42.819928,8.363779,0.063169,1458.822921,0.931784,0.710284
std,13.015376,23.307959,13.099142,26.217124,11.496298,94.380627,14.080941,11.813662,837.180475,0.252117,0.453631
min,-26.670000,0.000000,-26.670000,0.000000,0.000000,-179.974524,-83.230000,-52.850000,-297.000000,0.000000,0.000000
25%,-9.830000,32.590000,-10.100000,38.450000,10.067765,-0.631030,-0.070000,-8.800000,738.142857,1.000000,0.000000
50%,0.220000,49.630000,0.430000,57.220000,13.792103,62.017843,6.480000,0.210000,1470.444444,1.000000,1.000000
75%,10.770000,70.090000,11.220000,79.220000,18.781129,110.783936,12.960000,8.730000,2187.666667,1.000000,1.000000
max,26.660000,100.000000,26.660000,120.000000,101.211757,180.000000,99.070000,53.330000,2880.000000,1.000000,1.000000


In [4]:
model_bundle = train_etv_models(training_throws)
etv_model = build_etv_model(model_bundle)

In [5]:
Path("../models").mkdir(exist_ok=True)

save_model_bundle(model_bundle, "../models/all_games_baseline_etv_model.joblib")

'../models/all_games_baseline_etv_model.joblib'

In [6]:
# Train on many games above, then inspect one held-out/display game here.
display_game_id = "2024-06-08-NY-DC"

result = build_game_throws(
    game_id=display_game_id,
    etv_model=etv_model,
)

result.throws[
    ["thrower", "receiver", "completion", "xcp", "etv", "cpoe", "t_ec", "r_ec", "total_ec"]
].head(20)

,thrower,receiver,completion,xcp,etv,cpoe,t_ec,r_ec,total_ec
5,tedmonds,jtom,1,0.975129,0.579782,0.024871,0.013005,0.035807,0.048812
6,jtom,dbloodgoo,1,0.969610,0.613335,0.030390,-0.002254,0.044181,0.041927
7,dbloodgoo,cmccutche,1,0.959331,0.661362,0.040669,0.003846,0.059572,0.063419
8,cmccutche,tedmonds,1,0.978755,0.690290,0.021245,-0.030644,0.031120,0.000476
9,tedmonds,dbloodgoo,1,0.974197,0.678207,0.025803,-0.043204,0.037789,-0.005415
10,dbloodgoo,cjurek,1,0.950876,0.688779,0.049124,-0.027216,0.071875,0.044659
11,cjurek,tedmonds,1,0.976928,0.725309,0.023072,-0.035346,0.033763,-0.001583
12,tedmonds,cjurek,1,0.972800,0.706313,0.027200,-0.052759,0.039839,-0.012920
13,cjurek,tedmonds,1,0.968168,0.716961,0.031832,-0.029191,0.046560,0.017369
14,tedmonds,cjurek,1,0.755617,0.620604,0.244383,-0.142917,0.379396,0.236479


In [7]:
print("Display game:", result.game_id)
print("Throws with attached defender/block:", result.throws["defender"].notna().sum())
print("Box score total blocks:", result.box_score_stats["B"].sum())

result.box_score_stats.sort_values("B", ascending=False).head(40)

Display game: 2024-06-08-NY-DC
Throws with attached defender/block: 12
Box score total blocks: 12.0


,player,G,A,HA,T,B,C,CP%,HuR,HuCP%,...,total_t_aec,avg_t_aec,total_r_aec,avg_r_aec,total_total_aec,avg_total_aec,total_etv,avg_etv,expected_completions,completions_over_expected
17,adavis2,1.0,0,0.0,0,3.0,3,1.000000,0.000000,NaN,...,-0.141318,-0.047106,0.585888,0.195296,0.444570,0.148190,1.900071,0.633357,2.864090,0.135910
3,bjagt,2.0,2,0.0,0,2.0,19,1.000000,0.052632,1.000000,...,-2.951374,-0.155335,6.156002,0.324000,3.204629,0.168665,12.985366,0.683440,17.961387,1.038613
5,jmalks,1.0,4,1.0,2,1.0,27,0.931034,0.068966,1.000000,...,-2.226520,-0.076777,7.477736,0.257853,5.251217,0.181076,18.896001,0.651586,27.428805,-0.428805
1,cjurek,2.0,3,2.0,1,1.0,26,0.962963,0.037037,0.000000,...,-4.968628,-0.184023,8.258974,0.305888,3.290346,0.121865,19.096755,0.707287,25.577334,0.422666
13,jnissen,1.0,0,1.0,0,1.0,20,1.000000,0.000000,NaN,...,-3.134130,-0.156706,4.136039,0.206802,1.001910,0.050095,14.128238,0.706412,19.309649,0.690351
15,jtom,0.0,1,0.0,0,1.0,6,1.000000,0.000000,NaN,...,-0.461094,-0.076849,1.857967,0.309661,1.396873,0.232812,4.271495,0.711916,5.725838,0.274162
7,ebonnet,3.0,1,0.0,1,1.0,20,0.952381,0.000000,NaN,...,-1.914461,-0.091165,3.061402,0.145781,1.146941,0.054616,14.081239,0.670535,20.248997,-0.248997
6,tedmonds,0.0,3,1.0,0,1.0,24,1.000000,0.083333,1.000000,...,-3.557133,-0.148214,8.697742,0.362406,5.140609,0.214192,15.210239,0.633760,22.330916,1.669084
25,mling,0.0,0,0.0,0,1.0,2,1.000000,0.000000,NaN,...,-0.132562,-0.066281,0.127927,0.063963,-0.004635,-0.002317,1.165336,0.582668,1.965237,0.034763
0,jrandolph,3.0,3,3.0,0,0.0,36,1.000000,0.055556,1.000000,...,-5.864087,-0.162891,11.604178,0.322338,5.740091,0.159447,24.838076,0.689947,34.066275,1.933725


In [8]:
result.thrower_stats.head(20)

,thrower,attempts,completions,turnovers,huck_attempts,huck_completions,avg_throw_distance,avg_field_y_delta,total_field_y_delta,total_xcp,...,total_total_aec,avg_total_aec,total_etv,avg_etv,expected_completions,completions_over_expected,completion_pct,huck_rate,huck_completion_pct,cpoe
0,aagami,1,1,0,0,0,16.112359,-13.840000,-13.84,0.977640,...,-0.226739,-0.226739,0.682778,0.682778,0.977640,0.022360,1.000000,0.000000,NaN,0.022360
1,adavis2,3,3,0,0,0,17.009500,11.976667,35.93,2.864090,...,0.444570,0.148190,1.900071,0.633357,2.864090,0.135910,1.000000,0.000000,NaN,0.045303
2,amerriman,2,2,0,0,0,17.320931,3.915000,7.83,1.921159,...,0.121498,0.060749,1.295990,0.647995,1.921159,0.078841,1.000000,0.000000,NaN,0.039420
3,aolcese,1,1,0,0,0,22.791974,22.300000,22.30,0.946220,...,0.240016,0.240016,0.590824,0.590824,0.946220,0.053780,1.000000,0.000000,NaN,0.053780
4,aroy,33,31,2,2,0,17.322645,8.071818,266.37,31.152314,...,0.422218,0.012794,19.702523,0.597046,31.152314,-0.152314,0.939394,0.060606,0.0,-0.004616
5,bjagt,19,19,0,1,1,15.158649,7.843684,149.03,17.961387,...,3.204629,0.168665,12.985366,0.683440,17.961387,1.038613,1.000000,0.052632,1.0,0.054664
6,cjurek,27,26,1,1,0,13.890912,6.659630,179.81,25.577334,...,3.290346,0.121865,19.096755,0.707287,25.577334,0.422666,0.962963,0.037037,0.0,0.015654
7,cmccutche,8,8,0,0,0,10.818045,6.161250,49.29,7.716084,...,0.659811,0.082476,5.565146,0.695643,7.716084,0.283916,1.000000,0.000000,NaN,0.035490
8,cseamon,1,1,0,0,0,19.029149,8.590000,8.59,0.970955,...,0.088234,0.088234,0.523527,0.523527,0.970955,0.029045,1.000000,0.000000,NaN,0.029045
9,cweinberg,28,26,2,2,1,18.321000,9.474643,265.29,25.960698,...,3.215527,0.114840,18.682192,0.667221,25.960698,0.039302,0.928571,0.071429,0.5,0.001404


In [9]:
from ufa import (
    split_training_data,
    train_etv_models_from_split,
    evaluate_etv_model_bundle,
)

splits = split_training_data("../data/raw/all_games_1024.csv")

baseline_bundle = train_etv_models_from_split(splits)
baseline_results = evaluate_etv_model_bundle(baseline_bundle, splits)

baseline_results

,dataset,model,n,positive_rate,accuracy,auc,ppv,npv
0,train,cp,185093,0.938312,0.936583,0.788494,0.943015,0.431212
1,train,fv,186942,0.711317,0.711028,0.628491,0.712388,0.474766
2,validation,cp,46277,0.935584,0.934438,0.777733,0.940272,0.452080
3,validation,fv,46753,0.712104,0.711612,0.627445,0.712800,0.444444
4,temporal_test,cp,33762,0.942302,0.939755,0.793351,0.948062,0.418868
5,temporal_test,fv,34143,0.696248,0.694842,0.627865,0.697664,0.434426
6,player_test,cp,22804,0.942115,0.940186,0.777018,0.946397,0.417910
7,player_test,fv,22988,0.719027,0.718549,0.621423,0.720003,0.456693


In [12]:
from ufa import train_xgboost_etv_models_from_split

xgb_bundle = train_xgboost_etv_models_from_split(splits)

In [13]:
from ufa import compare_model_bundles, format_model_performance_table

comparison = compare_model_bundles(
    {"logistic": baseline_bundle, "xgboost": xgb_bundle},
    splits,
)

paper_table = format_model_performance_table(comparison)
paper_table

Column             FV Baseline  FV Model  CP Baseline  CP Model
Dataset  Metric                                                
Random   Accuracy        0.712     0.730        0.934     0.937
         AUC             0.627     0.656        0.778     0.802
         PPV             0.713     0.734        0.940     0.940
         NPV             0.444     0.664        0.452     0.627
Temporal Accuracy        0.695     0.712        0.940     0.944
         AUC             0.628     0.653        0.793     0.813
         PPV             0.698     0.717        0.948     0.946
         NPV             0.434     0.642        0.419     0.597
Player   Accuracy        0.719     0.736        0.940     0.943
         AUC             0.621     0.654        0.777     0.803
         PPV             0.720     0.742        0.946     0.945
         NPV             0.457     0.647        0.418     0.548
Train    Accuracy        0.711     0.728        0.937     0.941
         AUC             0.628     0.668        0.788     0.825
         PPV             0.712     0.732        0.943     0.943
         NPV             0.475     0.652        0.431     0.710

In [14]:
latex = model_performance_table_to_latex(paper_table)
print(latex)

\begin{table}
\centering
\caption{CP and FV Model performance compared to baseline.}
\begin{tabular}{llrrrr}
\toprule
 & Metric & FV Baseline & FV Model & CP Baseline & CP Model \\
\midrule
\multirow{4}{*}{Random} & Accuracy & 0.712 & 0.730 & 0.934 & 0.937 \\
 & AUC & 0.627 & 0.656 & 0.778 & 0.802 \\
 & PPV & 0.713 & 0.734 & 0.940 & 0.940 \\
 & NPV & 0.444 & 0.664 & 0.452 & 0.627 \\
\midrule
\multirow{4}{*}{Temporal} & Accuracy & 0.695 & 0.712 & 0.940 & 0.944 \\
 & AUC & 0.628 & 0.653 & 0.793 & 0.813 \\
 & PPV & 0.698 & 0.717 & 0.948 & 0.946 \\
 & NPV & 0.434 & 0.642 & 0.419 & 0.597 \\
\midrule
\multirow{4}{*}{Player} & Accuracy & 0.719 & 0.736 & 0.940 & 0.943 \\
 & AUC & 0.621 & 0.654 & 0.777 & 0.803 \\
 & PPV & 0.720 & 0.742 & 0.946 & 0.945 \\
 & NPV & 0.457 & 0.647 & 0.418 & 0.548 \\
\midrule
\multirow{4}{*}{Train} & Accuracy & 0.711 & 0.728 & 0.937 & 0.941 \\
 & AUC & 0.628 & 0.668 & 0.788 & 0.825 \\
 & PPV & 0.712 & 0.732 & 0.943 & 0.943 \\
 & NPV & 0.475 & 0.652 & 0.431 & 0.710 \